# 第一章：基本提示结构

- [课程](#lesson)
- [练习](#exercises)
- [示例游乐场](#example-playground)

## 设置

运行以下设置单元格以加载您的 API 密钥并建立 `get_completion` 辅助函数。

In [ ]:
!pip install anthropic

# 导入 Python 的内置正则表达式库
import re
import anthropic

# 从 IPython 存储中检索 API_KEY 和 MODEL_NAME 变量
%store -r API_KEY
%store -r MODEL_NAME

client = anthropic.Anthropic(api_key=API_KEY)

def get_completion(prompt: str, system_prompt=""):
    message = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt}
        ]
    )
    return message.content[0].text

---

## 课程

Anthropic 提供了两个 API，分别是旧版 [文本补全 API](https://docs.anthropic.com/claude/reference/complete_post) 和当前的 [消息 API](https://docs.anthropic.com/claude/reference/messages_post)。在本教程中，我们将仅使用消息 API。

使用消息 API 调用 Claude 至少需要以下参数：
- `model`：您打算调用的模型的 [API 模型名称](https://docs.anthropic.com/claude/docs/models-overview#model-recommendations)

- `max_tokens`：生成的最大令牌数，达到此限制后将停止。注意，Claude 可能在此最大值之前停止。此参数仅指定生成令牌的绝对最大数量。此外，这是一个 *硬性* 停止，意味着可能会导致 Claude 在生成单词或句子中间停止。

- `messages`：输入消息数组。我们的模型被训练为在交替的 `user` 和 `assistant` 对话轮次上运行。创建新 `Message` 时，您通过 messages 参数指定之前的对话轮次，然后模型生成对话中的下一个 `Message`。
  - 每个输入消息必须是一个包含 `role` 和 `content` 的对象。您可以指定单个 `user` 角色消息，也可以包含多个交替的 `user` 和 `assistant` 消息。第一个消息必须始终使用 `user` 角色。

还有一些可选参数，例如：
- `system`：系统提示 - 稍后会详细介绍。
  
- `temperature`：Claude 响应的可变性程度。在本课程和练习中，我们将 `temperature` 设置为 0。

有关所有 API 参数的完整列表，请访问我们的 [API 文档](https://docs.anthropic.com/claude/reference/messages_post)。

### 示例

让我们看看 Claude 如何响应一些格式正确的提示。对于以下每个单元格，运行单元格（`shift+enter`），Claude 的响应将显示在单元格下方。

In [ ]:
# 提示
PROMPT = "Hi Claude, how are you?"

# 打印 Claude 的响应
print(get_completion(PROMPT))

In [ ]:
# 提示
PROMPT = "Can you tell me the color of the ocean?"

# 打印 Claude 的响应
print(get_completion(PROMPT))

In [ ]:
# 提示
PROMPT = "What year was Celine Dion born in?"

# 打印 Claude 的响应
print(get_completion(PROMPT))

现在让我们看看一些不符合消息 API 正确格式的提示。对于这些格式错误的提示，消息 API 将返回错误。

首先，我们有一个消息 API 调用的示例，缺少 `messages` 数组中的 `role` 和 `content` 字段。

In [ ]:
# 获取 Claude 的响应
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"Hi Claude, how are you?"}
        ]
    )

# 打印 Claude 的响应
print(response[0].text)

这是一个没有在 `user` 和 `assistant` 角色之间交替的提示示例。

In [ ]:
# 获取 Claude 的响应
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# 打印 Claude 的响应
print(response[0].text)

`user` 和 `assistant` 消息 **必须交替**，并且消息 **必须以 `user` 轮次开始**。您可以在提示中包含多个 `user` 和 `assistant` 对（仿佛在模拟多轮对话）。您还可以在终端 `assistant` 消息中放入单词，让 Claude 从您停止的地方继续（更多内容将在后续章节中介绍）。

#### 系统提示

您还可以使用 **系统提示**。系统提示是一种在 “用户” 轮次提出问题或任务之前，向 Claude **提供上下文、指令和指南** 的方式。

在结构上，系统提示与 `user` 和 `assistant` 消息列表分开，因此属于单独的 `system` 参数（请查看笔记本 [设置](#setup) 部分中的 `get_completion` 辅助函数结构）。

在本教程中，无论何时可能使用系统提示，我们都在您的完成函数中提供了一个 `system` 字段。如果您不想使用系统提示，只需将 `SYSTEM_PROMPT` 变量设置为空字符串。

#### 系统提示示例

In [ ]:
# 系统提示
SYSTEM_PROMPT = "你的回答应始终是一系列推动对话的批判性思考问题（不要提供答案）。不要实际回答用户的问题。"

# 提示
PROMPT = "Why is the sky blue?"

# 打印 Claude 的响应
print(get_completion(PROMPT, SYSTEM_PROMPT))

为什么要使用系统提示？一个 **编写良好的系统提示可以提升 Claude 的表现**，例如增强 Claude 遵循规则和指令的能力。有关更多信息，请访问我们关于 [如何使用系统提示](https://docs.anthropic.com/claude/docs/how-to-use-system-prompts) 与 Claude 的文档。

现在我们将进入一些练习。如果您想在不更改上述内容的情况下试验课程提示，请滚动到本课程笔记本的底部，访问 [**示例游乐场**](#example-playground)。

---

## 练习
- [练习 1.1 - 数到三](#exercise-11---counting-to-three)
- [练习 1.2 - 系统提示](#exercise-12---system-prompt)

### 练习 1.1 - 数到三
使用正确的 `user` / `assistant` 格式，编辑下面的 `PROMPT`，让 Claude **数到三**。输出还将显示您的解决方案是否正确。

In [ ]:
# 提示 - 这是唯一需要更改的字段
PROMPT = "[Replace this text]"

# 获取 Claude 的响应
response = get_completion(PROMPT)

# 用于评分练习正确性的函数
def grade_exercise(text):
    pattern = re.compile(r'^(?=.*1)(?=.*2)(?=.*3).*$', re.DOTALL)
    return bool(pattern.match(text))

# 打印 Claude 的响应及其评分
print(response)
print("\n--------------------------- 评分 ---------------------------")
print("此练习已正确解决：", grade_exercise(response))

❓ 如果您需要提示，请运行下面的单元格！

In [ ]:
from hints import exercise_1_1_hint; print(exercise_1_1_hint)

### 练习 1.2 - 系统提示

修改 `SYSTEM_PROMPT`，使 Claude 像一个 3 岁的孩子一样回应。

In [ ]:
# 系统提示 - 这是唯一需要更改的字段
SYSTEM_PROMPT = "[Replace this text]"

# 提示
PROMPT = "How big is the sky?"

# 获取 Claude 的响应
response = get_completion(PROMPT, SYSTEM_PROMPT)

# 用于评分练习正确性的函数
def grade_exercise(text):
    return bool(re.search(r"giggles", text) or re.search(r"soo", text))

# 打印 Claude 的响应及其评分
print(response)
print("\n--------------------------- 评分 ---------------------------")
print("此练习已正确解决：", grade_exercise(response))

❓ 如果您需要提示，请运行下面的单元格！

In [ ]:
from hints import exercise_1_2_hint; print(exercise_1_2_hint)

### 恭喜！

如果您已经解决了所有练习，您可以进入下一章。祝您提示愉快！

---

## 示例游乐场

这是一个供您自由试验本课程中展示的提示示例的区域，您可以调整提示以查看它如何影响 Claude 的响应。

In [ ]:
# 提示
PROMPT = "Hi Claude, how are you?"

# 打印 Claude 的响应
print(get_completion(PROMPT))

In [ ]:
# 提示
PROMPT = "Can you tell me the color of the ocean?"

# 打印 Claude 的响应
print(get_completion(PROMPT))

In [ ]:
# 提示
PROMPT = "What year was Celine Dion born in?"

# 打印 Claude 的响应
print(get_completion(PROMPT))

In [ ]:
# 获取 Claude 的响应
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"Hi Claude, how are you?"}
        ]
    )

# 打印 Claude 的响应
print(response[0].text)

In [ ]:
# 获取 Claude 的响应
response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2000,
        temperature=0.0,
        messages=[
          {"role": "user", "content": "What year was Celine Dion born in?"},
          {"role": "user", "content": "Also, can you tell me some other facts about her?"}
        ]
    )

# 打印 Claude 的响应
print(response[0].text)

In [ ]:
# 系统提示
SYSTEM_PROMPT = "你的回答应始终是一系列推动对话的批判性思考问题（不要提供答案）。不要实际回答用户的问题。"

# 提示
PROMPT = "Why is the sky blue?"

# 打印 Claude 的响应
print(get_completion(PROMPT, SYSTEM_PROMPT))